# Question Data from Metaculus API

**Date:** 2026-02-11  
**Purpose:** Extract comprehensive question data via Metaculus API  
**Input:** `products/Run_Question_Map_2026-02-10_v01.csv`  
**Output:** `products/Question_Data_from_API_2026-02-11.csv`

This notebook:
1. Loads question list from run-question map
2. Fetches data from Metaculus API for each question
3. Extracts ~30+ fields per question
4. Saves to CSV with complete, authoritative data

**Advantages over HTML parsing:**
- ✅ Complete resolution values
- ✅ Forecaster/comment counts
- ✅ Authoritative status and dates
- ✅ Score/coverage data
- ✅ Community forecasts with intervals
- ✅ Stable API (won't break with HTML changes)

In [1]:
# Imports
import requests
import pandas as pd
import time
import json
from pathlib import Path
from datetime import date
from typing import Dict, Optional

print("✅ Imports successful")

✅ Imports successful


In [2]:
# Configuration
INPUT_FILE = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products/Run_Question_Map_2026-02-10_v01.csv")
OUTPUT_DIR = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products")
API_BASE = "https://www.metaculus.com/api2/questions"
TEST_LIMIT = None  #5  # Set to None for all questions
RATE_LIMIT_DELAY = 0.5  # Seconds between requests

print(f"Input file: {INPUT_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Test limit: {TEST_LIMIT}")
print(f"Rate limit: {RATE_LIMIT_DELAY}s between requests")

Input file: C:\Users\Donni\projects\metac_bot_Spring_2026\products\Run_Question_Map_2026-02-10_v01.csv
Output dir: C:\Users\Donni\projects\metac_bot_Spring_2026\products
Test limit: None
Rate limit: 0.5s between requests


In [3]:
# Load question list
df_runs = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df_runs)} run records")

# Extract unique question numbers (filter out empty values)
question_numbers = df_runs['question_number'].dropna().astype(int).unique().tolist()
question_numbers.sort()

print(f"Found {len(question_numbers)} unique questions")
print(f"Range: {min(question_numbers)} to {max(question_numbers)}")
print(f"First 10: {question_numbers[:10]}")

Loaded 1260 run records
Found 193 unique questions
Range: 41379 to 42078
First 10: [41379, 41380, 41382, 41383, 41384, 41385, 41386, 41389, 41392, 41396]


In [4]:
# API fetch function
def fetch_question_data(question_id: int) -> Optional[Dict]:
    """Fetch question data from Metaculus API."""
    url = f"{API_BASE}/{question_id}/"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"  ⚠️  API error: {e}")
        return None

print("✅ fetch_question_data() defined")

✅ fetch_question_data() defined


In [5]:
# Data extraction function
def extract_question_fields(data: Dict) -> Dict:
    """Extract relevant fields from API response."""
    
    # Top-level fields
    result = {
        'question_id': data.get('id'),
        'title': data.get('title', ''),
        'short_title': data.get('short_title', ''),
        'slug': data.get('slug', ''),
        'status': data.get('status', ''),
        'resolved': data.get('resolved', False),
        'comment_count': data.get('comment_count', 0),
        'nr_forecasters': data.get('nr_forecasters', 0),
        'forecasts_count': data.get('forecasts_count', 0),
        'author_username': data.get('author_username', ''),
        'curation_status': data.get('curation_status', ''),
    }
    
    # Dates
    result['created_at'] = data.get('created_at', '')
    result['published_at'] = data.get('published_at', '')
    result['edited_at'] = data.get('edited_at', '')
    result['open_time'] = data.get('open_time', '')
    result['actual_close_time'] = data.get('actual_close_time', '')
    result['scheduled_close_time'] = data.get('scheduled_close_time', '')
    result['actual_resolve_time'] = data.get('actual_resolve_time', '')
    result['scheduled_resolve_time'] = data.get('scheduled_resolve_time', '')
    
    # Question sub-object
    question = data.get('question', {})
    result['question_type'] = question.get('type', '')
    result['resolution'] = question.get('resolution')
    result['resolution_set_time'] = question.get('resolution_set_time', '')
    result['question_weight'] = question.get('question_weight', '')
    result['description'] = question.get('description', '')
    result['resolution_criteria'] = question.get('resolution_criteria', '')
    result['fine_print'] = question.get('fine_print', '')
    
    # Question-type specific fields
    if result['question_type'] == 'multiple_choice':
        result['mc_options'] = json.dumps(question.get('options', []))
    else:
        result['mc_options'] = ''
    
    if result['question_type'] == 'numeric':
        scaling = question.get('scaling', {})
        result['numeric_range_min'] = scaling.get('range_min', '')
        result['numeric_range_max'] = scaling.get('range_max', '')
        result['open_upper_bound'] = scaling.get('open_upper_bound', '')
        result['open_lower_bound'] = scaling.get('open_lower_bound', '')
    else:
        result['numeric_range_min'] = ''
        result['numeric_range_max'] = ''
        result['open_upper_bound'] = ''
        result['open_lower_bound'] = ''
    
    # Tournament info
    projects = data.get('projects', {})
    default_project = projects.get('default_project', {})
    result['tournament_id'] = default_project.get('id', '')
    result['tournament_name'] = default_project.get('name', '')
    result['tournament_slug'] = default_project.get('slug', '')
    
    # Community forecast from aggregations
    aggregations = data.get('aggregations', {})
    unweighted = aggregations.get('unweighted', {})
    latest = unweighted.get('latest', {})
    
    result['community_forecaster_count'] = latest.get('forecaster_count', '')
    forecast_values = latest.get('forecast_values', [])
    
    # Format community forecast based on type
    if result['question_type'] == 'binary' and len(forecast_values) == 2:
        result['community_forecast'] = f"{forecast_values[1]:.1%}"  # p_yes
        result['community_forecast_mean'] = forecast_values[1]
    elif result['question_type'] == 'multiple_choice':
        result['community_forecast'] = json.dumps(forecast_values)
        result['community_forecast_mean'] = ''
    elif result['question_type'] == 'numeric':
        means = latest.get('means', [])
        if means:
            result['community_forecast_mean'] = means[0]
            result['community_forecast'] = f"{means[0]:.2f}"
        else:
            result['community_forecast'] = ''
            result['community_forecast_mean'] = ''
    else:
        result['community_forecast'] = ''
        result['community_forecast_mean'] = ''
    
    # Interval bounds
    interval_lower = latest.get('interval_lower_bounds', [])
    interval_upper = latest.get('interval_upper_bounds', [])
    result['community_interval_lower'] = interval_lower[0] if interval_lower else ''
    result['community_interval_upper'] = interval_upper[0] if interval_upper else ''
    
    # Score data
    score_data = latest.get('score_data', {})
    result['coverage'] = score_data.get('coverage', '')
    result['peer_score'] = score_data.get('peer_score', '')
    result['baseline_score'] = score_data.get('baseline_score', '')
    result['spot_peer_score'] = score_data.get('spot_peer_score', '')
    result['spot_baseline_score'] = score_data.get('spot_baseline_score', '')
    
    return result

print("✅ extract_question_fields() defined")

✅ extract_question_fields() defined


In [6]:
# Test on single question
test_id = question_numbers[0]
print(f"Testing API fetch for question {test_id}...")

test_data = fetch_question_data(test_id)
if test_data:
    print(f"✅ API response received ({len(test_data)} top-level keys)")
    test_fields = extract_question_fields(test_data)
    print(f"✅ Extracted {len(test_fields)} fields")
    print("\nSample fields:")
    for k in ['question_id', 'title', 'question_type', 'status', 'resolved', 'resolution']:
        print(f"  {k}: {test_fields.get(k)}")
else:
    print("❌ API fetch failed")

Testing API fetch for question 41379...
✅ API response received (31 top-level keys)
✅ Extracted 44 fields

Sample fields:
  question_id: 41379
  title: Will the interest in “el salvador” change between 2026-01-05 and 2026-01-17 according to Google Trends?
  question_type: multiple_choice
  status: resolved
  resolved: True
  resolution: Doesn't change


In [7]:
# Batch fetch with rate limiting
questions_to_fetch = question_numbers[:TEST_LIMIT] if TEST_LIMIT else question_numbers
print(f"\nFetching {len(questions_to_fetch)} questions...\n")

results = []
failed = []

for i, qnum in enumerate(questions_to_fetch, 1):
    print(f"[{i}/{len(questions_to_fetch)}] Q{qnum}...", end='')
    
    data = fetch_question_data(qnum)
    if data:
        try:
            fields = extract_question_fields(data)
            results.append(fields)
            print(f" ✓ ({fields['question_type']}, {fields['status']})")
        except Exception as e:
            print(f" ⚠️  Extract failed: {e}")
            failed.append(qnum)
    else:
        print(f" ❌ Fetch failed")
        failed.append(qnum)
    
    # Rate limiting (be respectful)
    if i < len(questions_to_fetch):
        time.sleep(RATE_LIMIT_DELAY)

print(f"\n✅ Successfully fetched {len(results)} questions")
if failed:
    print(f"❌ Failed to fetch {len(failed)} questions: {failed}")


Fetching 193 questions...

 ✓ (multiple_choice, resolved)
[2/193] Q41380... ✓ (numeric, resolved)
[3/193] Q41382... ✓ (multiple_choice, resolved)
[4/193] Q41383... ✓ (numeric, resolved)
[5/193] Q41384... ✓ (multiple_choice, resolved)
[6/193] Q41385... ✓ (numeric, resolved)
[7/193] Q41386... ✓ (multiple_choice, resolved)
[8/193] Q41389... ✓ (multiple_choice, resolved)
[9/193] Q41392... ✓ (numeric, resolved)
[10/193] Q41396... ✓ (binary, resolved)
[11/193] Q41400...  ⚠️  API error: 429 Client Error: Too Many Requests for url: https://www.metaculus.com/api2/questions/41400/
 ❌ Fetch failed
[12/193] Q41403...  ⚠️  API error: 429 Client Error: Too Many Requests for url: https://www.metaculus.com/api2/questions/41403/
 ❌ Fetch failed
[13/193] Q41404...  ⚠️  API error: 429 Client Error: Too Many Requests for url: https://www.metaculus.com/api2/questions/41404/
 ❌ Fetch failed
[14/193] Q41406...  ⚠️  API error: 429 Client Error: Too Many Requests for url: https://www.metaculus.com/api2/questi

KeyboardInterrupt: 

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(results)
print(f"DataFrame shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}): {list(df.columns)}")
print(f"\nFirst 3 rows:")
df.head(3)

In [ ]:
# Data quality checks
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

print(f"\nQuestion Types:")
print(df['question_type'].value_counts())

print(f"\nStatus:")
print(df['status'].value_counts())

print(f"\nResolved:")
print(df['resolved'].value_counts())

print(f"\nResolution completeness (for resolved questions):")
resolved_df = df[df['resolved'] == True]
print(f"  Total resolved: {len(resolved_df)}")
print(f"  With resolution value: {resolved_df['resolution'].notna().sum()}")
print(f"  Missing resolution: {resolved_df['resolution'].isna().sum()}")

print(f"\nForecaster counts:")
print(f"  With nr_forecasters > 0: {(df['nr_forecasters'] > 0).sum()}")
print(f"  With community_forecaster_count > 0: {(df['community_forecaster_count'] != '').sum()}")

print(f"\nComment counts:")
print(f"  With comment_count > 0: {(df['comment_count'] > 0).sum()}")

print(f"\nCoverage/Score data:")
print(f"  With coverage: {(df['coverage'] != '').sum()}")
print(f"  With peer_score: {(df['peer_score'] != '').sum()}")

print(f"\nTournaments:")
print(df['tournament_name'].value_counts())

In [ ]:
# Save to CSV
output_file = OUTPUT_DIR / f"Question_Data_from_API_{date.today()}.csv"
df.to_csv(output_file, index=False)
print(f"\n✅ Saved to: {output_file}")
print(f"   Rows: {len(df)}")
print(f"   Columns: {len(df.columns)}")
print(f"   Size: {output_file.stat().st_size / 1024:.1f} KB")

In [ ]:
# Display sample of key fields
print("\n" + "=" * 80)
print("SAMPLE DATA (first 10 questions)")
print("=" * 80)

key_cols = ['question_id', 'short_title', 'question_type', 'status', 'resolved', 
            'resolution', 'nr_forecasters', 'community_forecast']
available_cols = [c for c in key_cols if c in df.columns]
df[available_cols].head(10)